In [18]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.preprocessing import MinMaxScaler, LabelEncoder

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.svm import SVC

from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import VotingClassifier

from sklearn.metrics import accuracy_score


In [19]:

data1 = pd.read_csv("Iris_Data.csv")
data1

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


In [20]:

le = LabelEncoder()

data1["species"] = le.fit_transform(data1["species"])

In [21]:

X = data1.drop("species", axis=1)

y = data1["species"]

In [22]:
X

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [23]:
y

0      0
1      0
2      0
3      0
4      0
      ..
145    2
146    2
147    2
148    2
149    2
Name: species, Length: 150, dtype: int64

In [24]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [25]:

clf1 = Pipeline([
    ("scaler", MinMaxScaler()),
    ("model", LogisticRegression(max_iter=1000))
])


In [26]:

clf2 = Pipeline([
    ("model", DecisionTreeClassifier(max_depth=4, random_state=42))
])


In [27]:

clf3 = Pipeline([
    ("scaler", MinMaxScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])


In [28]:

from sklearn.calibration import CalibratedClassifierCV

In [29]:

clf4 = Pipeline([
    ("scaler", MinMaxScaler()),
    ("model", CalibratedClassifierCV(
        estimator=SVC(kernel="rbf", random_state=42),
        ensemble=False
    ))
])


In [30]:

classifiers = [
    ("LogReg", clf1),
    ("DecisionTree", clf2),
    ("KNN", clf3),
    ("SVM", clf4)
]

In [31]:
classifiers

[('LogReg',
  Pipeline(steps=[('scaler', MinMaxScaler()),
                  ('model', LogisticRegression(max_iter=1000))])),
 ('DecisionTree',
  Pipeline(steps=[('model',
                   DecisionTreeClassifier(max_depth=4, random_state=42))])),
 ('KNN',
  Pipeline(steps=[('scaler', MinMaxScaler()), ('model', KNeighborsClassifier())])),
 ('SVM',
  Pipeline(steps=[('scaler', MinMaxScaler()),
                  ('model',
                   CalibratedClassifierCV(ensemble=False,
                                          estimator=SVC(random_state=42)))]))]

In [32]:

results = {}

for name, clf in classifiers:
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    acc = accuracy_score(y_test, pred)
    cv = cross_val_score(clf, X_train, y_train, cv=5)

    results[name] = (acc, cv.mean())

    print(f"\n{name}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"CV Accuracy: {cv.mean():.4f} ± {cv.std():.4f}")
    


LogReg
Test Accuracy: 0.8667
CV Accuracy: 0.9524 ± 0.0301

DecisionTree
Test Accuracy: 0.8889
CV Accuracy: 0.9429 ± 0.0190

KNN
Test Accuracy: 0.9333
CV Accuracy: 0.9714 ± 0.0233

SVM
Test Accuracy: 0.9333
CV Accuracy: 0.9714 ± 0.0233


In [33]:

voting_hard = VotingClassifier(estimators=classifiers, voting="hard" )

voting_hard.fit(X_train, y_train)

pred_hard = voting_hard.predict(X_test)

acc_hard = accuracy_score(y_test, pred_hard)

print("\nHard Voting Accuracy:", acc_hard)


Hard Voting Accuracy: 0.9111111111111111


In [34]:

voting_soft = VotingClassifier(
    estimators=classifiers,
    voting="soft"
)

voting_soft.fit(X_train, y_train)

pred_soft = voting_soft.predict(X_test)

acc_soft = accuracy_score(y_test, pred_soft)

print("\nSoft Voting Accuracy:", acc_soft)



Soft Voting Accuracy: 0.9333333333333333
